In [1]:
!pip install duckdb

In [6]:
import duckdb
con = duckdb.connect()

# ── YELLOW ──────────────────────────────────────────
yellow = con.execute("""
    SELECT
        COUNT(*) as total_original,
        SUM(CASE WHEN fare_amount < 0 AND payment_type IN (0, 1, 2) 
            THEN 1 ELSE 0 END) as eliminar_fare_negativo,
        SUM(CASE WHEN trip_distance <= 0 
            AND (epoch(tpep_dropoff_datetime) - epoch(tpep_pickup_datetime))/60.0 > 3
            THEN 1 ELSE 0 END) as eliminar_distancia_gps,
        SUM(CASE WHEN fare_amount > 300 THEN 1 ELSE 0 END) as eliminar_fare_extremo,
        SUM(CASE WHEN trip_distance > 80 THEN 1 ELSE 0 END) as eliminar_distancia_extrema,
        SUM(CASE WHEN (epoch(tpep_dropoff_datetime) - epoch(tpep_pickup_datetime))/60.0 < 1
            THEN 1 ELSE 0 END) as eliminar_viaje_fantasma,
        SUM(CASE WHEN (epoch(tpep_dropoff_datetime) - epoch(tpep_pickup_datetime))/60.0 > 200
            THEN 1 ELSE 0 END) as eliminar_duracion_extrema,
        SUM(CASE WHEN YEAR(tpep_pickup_datetime) NOT IN (2023, 2024, 2025)
            THEN 1 ELSE 0 END) as eliminar_fecha_corrupta,
        SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) as eliminar_sin_origen,
        SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) as eliminar_sin_destino
    FROM read_parquet('/home/jovyan/data/bronze/yellow/2023/yellow_tripdata_2023-01.parquet')
""").df()

# ── GREEN ──────────────────────────────────────────
green = con.execute("""
    SELECT
        COUNT(*) as total_original,
        SUM(CASE WHEN fare_amount < 0 AND payment_type IN (0, 1, 2) 
            THEN 1 ELSE 0 END) as eliminar_fare_negativo,
        SUM(CASE WHEN trip_distance <= 0 
            AND (epoch(lpep_dropoff_datetime) - epoch(lpep_pickup_datetime))/60.0 > 3
            THEN 1 ELSE 0 END) as eliminar_distancia_gps,
        SUM(CASE WHEN fare_amount > 300 THEN 1 ELSE 0 END) as eliminar_fare_extremo,
        SUM(CASE WHEN fare_amount < 2.50 AND fare_amount >= 0
        THEN 1 ELSE 0 END) as eliminar_fare_minimo,
        SUM(CASE WHEN trip_distance > 80 THEN 1 ELSE 0 END) as eliminar_distancia_extrema,
        SUM(CASE WHEN (epoch(lpep_dropoff_datetime) - epoch(lpep_pickup_datetime))/60.0 < 1
            THEN 1 ELSE 0 END) as eliminar_viaje_fantasma,
        SUM(CASE WHEN (epoch(lpep_dropoff_datetime) - epoch(lpep_pickup_datetime))/60.0 > 200
            THEN 1 ELSE 0 END) as eliminar_duracion_extrema,
        SUM(CASE WHEN YEAR(lpep_pickup_datetime) NOT IN (2023, 2024, 2025)
            THEN 1 ELSE 0 END) as eliminar_fecha_corrupta,
        SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) as eliminar_sin_origen,
        SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) as eliminar_sin_destino
    FROM read_parquet('/home/jovyan/data/bronze/green/2023/green_tripdata_2023-01.parquet')
""").df()

# ── FHV ──────────────────────────────────────────
fhv = con.execute("""
    SELECT
        COUNT(*) as total_original,
        SUM(CASE WHEN PUlocationID IS NULL THEN 1 ELSE 0 END) as eliminar_sin_origen,
        SUM(CASE WHEN DOlocationID IS NULL THEN 1 ELSE 0 END) as eliminar_sin_destino,
        SUM(CASE WHEN (epoch(dropOff_datetime) - epoch(pickup_datetime))/60.0 < 1
            THEN 1 ELSE 0 END) as eliminar_viaje_fantasma,
        SUM(CASE WHEN (epoch(dropOff_datetime) - epoch(pickup_datetime))/60.0 > 200
            THEN 1 ELSE 0 END) as eliminar_duracion_extrema,
        SUM(CASE WHEN YEAR(pickup_datetime) NOT IN (2023, 2024, 2025)
            THEN 1 ELSE 0 END) as eliminar_fecha_corrupta
    FROM read_parquet('/home/jovyan/data/bronze/fhv/2023/fhv_tripdata_2023-01.parquet')
""").df()

# ── FHVHV ──────────────────────────────────────────
fhvhv = con.execute("""
    SELECT
        COUNT(*) as total_original,
        SUM(CASE WHEN base_passenger_fare < 0 THEN 1 ELSE 0 END) as eliminar_fare_negativo,
        SUM(CASE WHEN trip_miles <= 0 
            AND (epoch(dropoff_datetime) - epoch(pickup_datetime))/60.0 > 3
            THEN 1 ELSE 0 END) as eliminar_distancia_gps,
        SUM(CASE WHEN (epoch(dropoff_datetime) - epoch(pickup_datetime))/60.0 < 1
            THEN 1 ELSE 0 END) as eliminar_viaje_fantasma,
        SUM(CASE WHEN (epoch(dropoff_datetime) - epoch(pickup_datetime))/60.0 > 200
            THEN 1 ELSE 0 END) as eliminar_duracion_extrema,
        SUM(CASE WHEN YEAR(pickup_datetime) NOT IN (2023, 2024, 2025)
            THEN 1 ELSE 0 END) as eliminar_fecha_corrupta,
        SUM(CASE WHEN base_passenger_fare < 2.50 AND base_passenger_fare >= 0
            THEN 1 ELSE 0 END) as eliminar_fare_minimo,
        SUM(CASE WHEN driver_pay <= 0
            THEN 1 ELSE 0 END) as eliminar_driver_pay_invalido,
        SUM(CASE WHEN trip_miles > 150
            THEN 1 ELSE 0 END) as eliminar_distancia_extrema
    FROM read_parquet('/home/jovyan/data/bronze/fhvhv/2023/fhvhv_tripdata_2023-01.parquet')
""").df()

# ── IMPRIMIR RESULTADOS ──────────────────────────────────────────
def imprimir_reporte(df, nombre):
    total = int(df["total_original"].values[0])
    print(f"\n{'='*50}")
    print(f"  {nombre}")
    print(f"{'='*50}")
    print(f"  Total filas: {total:,}")
    print(f"  {'-'*46}")
    for col in df.columns:
        if col != "total_original":
            val = int(df[col].values[0])
            pct = val / total * 100
            if pct == 0:
                estado = "OK"
            elif pct <= 1:
                estado = "BAJO"
            elif pct <= 5:
                estado = "REVISAR"
            else:
                estado = "CRITICO"
            print(f"  [{estado}] {col}: {val:,} ({pct:.2f}%)")
    print(f"{'='*50}")

imprimir_reporte(yellow, "YELLOW TAXI - enero 2023")
imprimir_reporte(green,  "GREEN TAXI - enero 2023")
imprimir_reporte(fhv,    "FHV - enero 2023")
imprimir_reporte(fhvhv,  "FHVHV (Uber/Lyft) - enero 2023")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


  YELLOW TAXI - enero 2023
  Total filas: 3,066,766
  ----------------------------------------------
  [BAJO] eliminar_fare_negativo: 5,766 (0.19%)
  [BAJO] eliminar_distancia_gps: 20,501 (0.67%)
  [BAJO] eliminar_fare_extremo: 213 (0.01%)
  [BAJO] eliminar_distancia_extrema: 114 (0.00%)
  [REVISAR] eliminar_viaje_fantasma: 33,335 (1.09%)
  [BAJO] eliminar_duracion_extrema: 3,005 (0.10%)
  [BAJO] eliminar_fecha_corrupta: 38 (0.00%)
  [OK] eliminar_sin_origen: 0 (0.00%)
  [OK] eliminar_sin_destino: 0 (0.00%)

  GREEN TAXI - enero 2023
  Total filas: 68,211
  ----------------------------------------------
  [OK] eliminar_fare_negativo: 0 (0.00%)
  [REVISAR] eliminar_distancia_gps: 2,079 (3.05%)
  [BAJO] eliminar_fare_extremo: 13 (0.02%)
  [BAJO] eliminar_fare_minimo: 96 (0.14%)
  [BAJO] eliminar_distancia_extrema: 37 (0.05%)
  [REVISAR] eliminar_viaje_fantasma: 1,645 (2.41%)
  [BAJO] eliminar_duracion_extrema: 263 (0.39%)
  [BAJO] eliminar_fecha_corrupta: 3 (0.00%)
  [OK] eliminar_sin_o